# SimpleGPT - Training on Google Colab
Train a decoder-only GPT from scratch using the project's own modules.

## 1. Mount Google Drive (optional, for saving checkpoints)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Clone the repo

In [2]:
!git clone https://github.com/mrroy-dev/simple-chatbot.git
%cd simple-chatbot

fatal: destination path 'simple-chatbot' already exists and is not an empty directory.
/content/simple-chatbot


## 3. Install dependencies

In [3]:
!pip install torch pyyaml tqdm


## 4. Check GPU

In [4]:
import torch
# IMPORTANT: If you get a CUDA error, do Runtime -> Restart runtime
try:
    torch.cuda.synchronize()
except Exception:
    print('CUDA error detected from previous run. Restart runtime and re-run.')
    raise

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

Using device: cuda
GPU: Tesla T4
Memory: 15.64 GB


## 5. Load or generate training data

In [5]:
import json, os, random, sys
os.makedirs('data', exist_ok=True)
random.seed(42)

data_path = 'data/pairs_train.json'
if os.path.exists(data_path):
    print(f'Loading {data_path}...')
    with open(data_path, 'r') as f:
        all_pairs = json.load(f)
    print(f'Total entries: {len(all_pairs)}')
    random.shuffle(all_pairs)
    MAX_EXAMPLES = min(30000, len(all_pairs))
    all_pairs = all_pairs[:MAX_EXAMPLES]
else:
    print(f'{data_path} not found. Generating synthetic data...')
    facts = {
        'python': 'Python is a high-level programming language known for readability.',
        'ml': 'Machine learning enables systems to learn from data without explicit programming.',
        'dl': 'Deep learning uses multi-layered neural networks to learn from data.',
        'nn': 'Neural networks are computing systems inspired by biological brains.',
        'transformers': 'Transformers use self-attention to process sequential data in parallel.',
        'gpt': 'GPT is a language model that generates human-like text.',
        'pytorch': 'PyTorch is an open-source ML framework by Meta AI.',
        'gd': 'Gradient descent minimizes loss by iteratively updating parameters.',
        'overfitting': 'Overfitting occurs when a model memorizes noise in training data.',
        'attention': 'Attention lets models focus on relevant parts of input.',
        'nlp': 'NLP enables computers to understand and generate human language.',
        'cnn': 'CNNs use convolutional layers to process grid-like data such as images.',
        'rnn': 'RNNs process sequences by maintaining a hidden state across time steps.',
        'lstm': 'LSTM networks solve vanishing gradients using gated cell states.',
        'gan': 'GANs use a generator and discriminator that compete to create realistic data.',
        'dropout': 'Dropout prevents overfitting by randomly dropping neurons during training.',
        'embedding': 'Embeddings map discrete tokens to dense vector representations.',
        'supervised': 'Supervised learning trains on labeled input-output pairs.',
        'rl': 'Reinforcement learning trains agents via rewards and penalties.',
        'backprop': 'Backpropagation computes gradients using the chain rule.',
    }
    questions = ['What is {t}?', 'Explain {t}', 'Tell me about {t}', 'Describe {t}', 'How does {t} work?']
    all_pairs = []
    for topic, fact in facts.items():
        q = random.choice(questions).format(t=topic)
        all_pairs.append({'context': f'<bos> <user> {q}', 'response': fact})
    for _ in range(500):
        t = random.choice(list(facts.keys()))
        q = random.choice(questions).format(t=t)
        all_pairs.append({'context': f'<bos> <user> {q}', 'response': facts[t]})
    print(f'Generated {len(all_pairs)} synthetic pairs')

split = int(len(all_pairs) * 0.95)
train = all_pairs[:split]
val = all_pairs[split:]
with open('data/pairs_train_subset.json', 'w') as f:
    json.dump(train, f)
with open('data/pairs_val_subset.json', 'w') as f:
    json.dump(val, f)
print(f'Using {len(train)} train + {len(val)} val pairs')


data/pairs_train.json not found. Generating synthetic data...
Generated 520 synthetic pairs
Using 494 train + 26 val pairs


## 6. Import project modules
Uses the project's own BPETokenizer (with byte_encoder fix), ModelConfig, SimpleGPT (GQA, SwiGLU, RoPE, RMSNorm), and ConversationDataset.

In [6]:
sys.path.insert(0, '/content/simple-chatbot')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from simple_gpt import ModelConfig, SimpleGPT, BPETokenizer
from simple_gpt.datasets.dataset import ConversationDataset

print('Imports OK')

Imports OK


## 7. Build tokenizer on the training data

In [7]:
# Collect all text for training the BPE tokenizer
all_texts = []
for item in train:
    all_texts.append(item['context'])
    all_texts.append(item['response'])

tok = BPETokenizer()
tok._build_vocab(all_texts, min_freq=2, max_vocab_size=8000)
print(f'Tokenizer vocab: {len(tok.vocab)} tokens')
print(f'Special IDs: {tok.special_ids}')
print(f'PAD={tok.pad_id()} BOS={tok.bos_id()} EOS={tok.eos_id()}')

Tokenizer vocab: 672 tokens
Special IDs: {'pad': 256, 'unk': 257, 'bos': 274, 'eos': 259, 'user': 260, 'bot': 261, 'system': 262, 'tool': 263}
PAD=256 BOS=274 EOS=259


## 8. Create model using project's ModelConfig

In [8]:
config = ModelConfig(
    vocab_size=len(tok.vocab),
    block_size=256,
    n_layer=6,
    n_head=8,
    n_kv_head=4,
    n_embd=384,
    dropout=0.1,
    embd_dropout=0.1,
    weight_tying=True,
)
print(f'Model params: {sum(p.numel() for p in SimpleGPT(config).parameters()):,}')

model = SimpleGPT(config).to(device)
print(f'Model on {device} (vocab_size={config.vocab_size})')

Model params: 9,995,136
Model on cuda (vocab_size=672)


## 9. Create datasets and data loaders

In [9]:
BATCH_SIZE = 32

train_ds = ConversationDataset(train, tok, seq_len=config.block_size)
val_ds = ConversationDataset(val, tok, seq_len=config.block_size)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train: {len(train_ds)} examples, {len(train_loader)} batches')
print(f'Val: {len(val_ds)} examples, {len(val_loader)} batches')

Train: 494 examples, 15 batches
Val: 26 examples, 1 batches


## 10. Training setup
AdamW with weight decay, warmup-cosine LR scheduler.

In [10]:
import math

LR = 3e-4
WARMUP_STEPS = 50
MAX_STEPS = 1000
LOG_EVERY = 10
VAL_EVERY = 100

# AdamW with separate weight decay
decay_params = []
no_decay_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.ndim < 2 or 'bias' in name or 'norm' in name or 'ln' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=LR, betas=(0.9, 0.95), eps=1e-8, fused=True)

def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(MAX_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1 + math.cos(math.pi * progress))

# Cosine schedule (will be updated per step)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

print('Optimizer & scheduler ready')

Optimizer & scheduler ready


## 11. Training loop

In [12]:
from tqdm.notebook import tqdm

global_step = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []

model.train()
pbar = tqdm(total=MAX_STEPS, desc='Training')

while global_step < MAX_STEPS:
    for batch in train_loader:
        if global_step >= MAX_STEPS:
            break

        x, y, m, _ = batch
        x, y, m = x.to(device), y.to(device), m.to(device)

        logits, loss, _ = model(x, targets=y, loss_mask=m)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        global_step += 1
        pbar.update(1)

        if global_step % LOG_EVERY == 0:
            train_losses.append((global_step, loss.item()))
            pbar.set_postfix(loss=f'{loss.item():.4f}')

        if global_step % VAL_EVERY == 0:
            model.eval()
            val_loss_total = 0.0
            val_batches = 0
            with torch.no_grad():
                for vx, vy, vm, _ in val_loader:
                    vx, vy, vm = vx.to(device), vy.to(device), vm.to(device)
                    _, vloss, _ = model(vx, targets=vy, loss_mask=vm)
                    val_loss_total += vloss.item()
                    val_batches += 1
            val_loss = val_loss_total / max(val_batches, 1)
            val_losses.append((global_step, val_loss))

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    'model_state': model.state_dict(),
                    'model_config': config,
                    'tokenizer': tok,
                    'step': global_step,
                    'val_loss': val_loss,
                }, 'checkpoint_best.pt')
                print(f'Step {global_step}: new best val_loss={val_loss:.4f} (saved)')
            else:
                print(f'Step {global_step}: val_loss={val_loss:.4f}')

            model.train()

pbar.close()
print(f'Done! Best val_loss: {best_val_loss:.4f}')

# Save final checkpoint
torch.save({
    'model_state': model.state_dict(),
    'model_config': config,
    'tokenizer': tok,
    'step': global_step,
    'final_loss': loss.item(),
}, 'checkpoint_final.pt')
print('Final checkpoint saved')

Training:   0%|          | 0/1000 [00:00<?, ?it/s]

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 12. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
if train_losses:
    steps, losses = zip(*train_losses)
    plt.plot(steps, losses)
plt.xlabel('Step')
plt.ylabel('Train Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
if val_losses:
    steps, losses = zip(*val_losses)
    plt.plot(steps, losses, 'r-o')
plt.xlabel('Step')
plt.ylabel('Val Loss')
plt.title('Validation Loss')
plt.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()
print(f'Final train loss: {train_losses[-1][1]:.4f}  Best val loss: {best_val_loss:.4f}')

## 13. Load best checkpoint and chat

In [ ]:
# Load best checkpoint
ckpt = torch.load('checkpoint_best.pt', map_location=device)
model = SimpleGPT(ckpt['model_config']).to(device)
model.load_state_dict(ckpt['model_state'])
tok = ckpt['tokenizer']
model.eval()
print(f'Loaded best checkpoint from step {ckpt["step"]} (val_loss={ckpt["val_loss"]:.4f})')

## 14. Chat function

In [ ]:
@torch.no_grad()
def generate(model, tok, prompt, max_new=100, temp=0.7, top_k=40, top_p=0.9, rep_penalty=1.2):
    input_ids = [tok.bos_id()] + tok.encode(prompt) + [tok.bot_id()]
    idx = torch.tensor([input_ids], dtype=torch.long, device=device)

    for _ in range(max_new):
        if idx.size(1) > model.block_size:
            idx = idx[:, -model.block_size:]
        logits, _, _ = model(idx)
        logits = logits[:, -1, :] / max(temp, 1e-5)

        if rep_penalty is not None:
            for tid in set(idx[0].tolist()):
                logits[0, tid] /= rep_penalty

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_mask = cum_probs > top_p
            sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
            sorted_mask[..., 0] = 0
            indices_to_remove = sorted_mask.scatter(1, sorted_indices, sorted_mask)
            logits[indices_to_remove] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

        if next_id.item() == tok.eos_id():
            break

    return tok.decode(idx[0].tolist())


def chat():
    print('\n=== SimpleGPT Chat ===')
    print('Type "quit" to exit, "clear" to reset history\n')
    history = []
    while True:
        user = input('You: ').strip()
        if user.lower() == 'quit':
            break
        if user.lower() == 'clear':
            history = []
            print('History cleared')
            continue
        if not user:
            continue
        context = ' '.join(history[-4:])
        prompt = f'<bos> <user> {user}' if not context else f'<bos> <user> {context} <user> {user}'
        response = generate(model, tok, prompt, max_new=100)
        print(f'Bot: {response}')
        history.extend([f'<user> {user}', f'<bot> {response}'])


chat()

## 15. Export to Google Drive

In [ ]:
import shutil
shutil.copy('checkpoint_best.pt', '/content/drive/MyDrive/') if os.path.exists('checkpoint_best.pt') else None
shutil.copy('checkpoint_final.pt', '/content/drive/MyDrive/') if os.path.exists('checkpoint_final.pt') else None
print('Copied to Drive' if os.path.exists('/content/drive') else 'Drive not mounted, skip export')